# 🎙️ Servidor do Narrador

Este é o "motor" que gera a sua voz. A sua voz e o histórico ficam guardados na sua página do GitHub; aqui só acontece a narração.

**Como usar:** no menu, clique em **Ambiente de execução → Executar tudo** (ou Ctrl+F9). Em uns 5 minutos aparece um link **"Clique aqui para abrir o seu narrador já conectado"**. Deixe esta aba aberta enquanto narra.

Se aparecer um aviso pedindo para **reiniciar a sessão**, aceite e clique em **Executar tudo** de novo.

In [ ]:
# Passo 1: instalar (2 a 3 minutos)
!pip install -q coqui-tts gradio soundfile

In [ ]:
# Passo 2: ligar o servidor (na primeira vez baixa o modelo de voz, ~2 GB)
"""
Servidor do Narrador com a sua voz (roda no Google Colab, com GPU grátis).

Recebe uma ou mais gravações da sua voz + o texto da história e devolve
um arquivo WAV narrado com a sua voz, usando o modelo XTTS v2.
A página index.html (no GitHub Pages) conversa com este servidor pela API
"/narrar".
"""

import hashlib
import os
import re
import tempfile
import time

os.environ.setdefault("COQUI_TOS_AGREED", "1")  # licença não comercial do XTTS v2

import numpy as np
import soundfile as sf
import torch
import gradio as gr
from TTS.api import TTS
from urllib.parse import quote

PAGINA = "https://fonsecaalari-gif.github.io/narrador/"


# Lê as gravações com soundfile (evita problemas do torchaudio novo no Colab).
def _load_audio_sf(audiopath, sampling_rate):
    import torchaudio.functional as AF
    data, sr = sf.read(audiopath, dtype="float32", always_2d=True)
    audio = torch.from_numpy(data.mean(axis=1)).unsqueeze(0)
    if sr != sampling_rate:
        audio = AF.resample(audio, sr, sampling_rate)
    return audio.clamp(-1, 1)


try:
    import TTS.tts.models.xtts as _xtts_mod
    _xtts_mod.load_audio = _load_audio_sf
except Exception as e:  # noqa: BLE001
    print("Aviso: não consegui trocar o leitor de áudio:", e)

torch.set_num_threads(max(1, os.cpu_count() or 1))

SAMPLE_RATE = 24000
MAX_CHARS = 200          # tamanho de cada trecho enviado ao modelo
LIMITE_TEXTO = 30000     # caracteres por história
IDIOMA = "pt"

print("Carregando o modelo XTTS v2...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device).synthesizer.tts_model
print(f"Modelo pronto em {device}.")

_cache_voz = {}


def _hash_arquivos(caminhos):
    h = hashlib.sha256()
    for c in sorted(caminhos):
        with open(c, "rb") as f:
            h.update(hashlib.sha256(f.read()).digest())
    return h.hexdigest()


def extrair_voz(caminhos):
    chave = _hash_arquivos(caminhos)
    if chave not in _cache_voz:
        if len(_cache_voz) > 20:
            _cache_voz.clear()
        _cache_voz[chave] = model.get_conditioning_latents(audio_path=list(caminhos))
    return _cache_voz[chave]


# ------------------------------------------------------------------ texto
def _quebrar_frase_longa(frase):
    partes, atual = [], ""
    for pedaco in re.split(r"(?<=[,;:])\s+", frase):
        palavras = pedaco.split() if len(pedaco) > MAX_CHARS else [pedaco]
        for p in palavras:
            if len(atual) + len(p) + 1 > MAX_CHARS and atual:
                partes.append(atual.strip())
                atual = ""
            atual += p + " "
    if atual.strip():
        partes.append(atual.strip())
    return partes


def preparar_texto(texto):
    """Lista de parágrafos; cada parágrafo é uma lista de trechos curtos."""
    paragrafos = [p.strip() for p in re.split(r"\n\s*\n", texto) if p.strip()]
    resultado = []
    for p in paragrafos:
        p = re.sub(r"\s+", " ", p)
        trechos, atual = [], ""
        for f in re.split(r"(?<=[.!?…])\s+", p):
            if len(f) > MAX_CHARS:
                if atual.strip():
                    trechos.append(atual.strip())
                atual = ""
                trechos.extend(_quebrar_frase_longa(f))
            elif len(atual) + len(f) + 1 > MAX_CHARS and atual:
                trechos.append(atual.strip())
                atual = f + " "
            else:
                atual += f + " "
        if atual.strip():
            trechos.append(atual.strip())
        resultado.append(trechos)
    return resultado


def _silencio(segundos):
    return np.zeros(int(SAMPLE_RATE * float(segundos)), dtype=np.float32)


def _caminho(f):
    if f is None:
        return None
    if isinstance(f, str):
        return f
    if isinstance(f, dict):
        return f.get("path") or f.get("name")
    return getattr(f, "path", None) or getattr(f, "name", None)


# ------------------------------------------------------------------ narração
def narrar(vozes, texto, velocidade=1.0, expressividade=0.7,
           pausa_frase=0.25, pausa_paragrafo=0.9):
    caminhos = [c for c in (_caminho(v) for v in (vozes or [])) if c]
    if not caminhos:
        raise gr.Error("Nenhuma gravação de voz foi enviada.")
    texto = (texto or "").strip()
    if not texto:
        raise gr.Error("O texto da história está vazio.")
    if len(texto) > LIMITE_TEXTO:
        raise gr.Error(f"A história tem mais de {LIMITE_TEXTO} caracteres. Divida em partes.")

    yield "Aprendendo a sua voz...", None
    gpt_latent, speaker_emb = extrair_voz(caminhos)

    paragrafos = preparar_texto(texto)
    total = sum(len(p) for p in paragrafos)
    feitos, audio, inicio = 0, [], time.time()

    for i, trechos in enumerate(paragrafos):
        for j, trecho in enumerate(trechos):
            if feitos == 0:
                msg = f"Narrando trecho 1 de {total}..."
            else:
                restante = (time.time() - inicio) / feitos * (total - feitos)
                msg = f"Narrando trecho {feitos + 1} de {total} (faltam ~{max(1, round(restante / 60))} min)"
            yield msg, None

            out = model.inference(
                trecho, IDIOMA, gpt_latent, speaker_emb,
                temperature=float(expressividade),
                speed=float(velocidade),
                enable_text_splitting=False,
            )
            audio.append(np.asarray(out["wav"], dtype=np.float32))
            if j < len(trechos) - 1:
                audio.append(_silencio(pausa_frase))
            feitos += 1
        if i < len(paragrafos) - 1:
            audio.append(_silencio(pausa_paragrafo))

    final = np.concatenate(audio)
    pico = float(np.max(np.abs(final))) or 1.0
    final = (final / pico * 0.95).astype(np.float32)

    pasta = tempfile.mkdtemp(prefix="narracao_")
    arquivo = os.path.join(pasta, "narracao.wav")
    sf.write(arquivo, final, SAMPLE_RATE, subtype="PCM_16")
    yield "Pronto!", arquivo


# ------------------------------------------------------------------ interface
with gr.Blocks(title="Narrador com a sua voz — servidor") as demo:
    gr.Markdown(
        "# Servidor do Narrador\n"
        "Este é o servidor. Use a sua página do GitHub para narrar e guardar o histórico, "
        "ou teste aqui mesmo."
    )
    vozes = gr.File(file_count="multiple", file_types=["audio"], label="Gravações da sua voz")
    texto = gr.Textbox(lines=8, label="História")
    with gr.Row():
        velocidade = gr.Slider(0.7, 1.3, value=1.0, step=0.05, label="Velocidade")
        expressividade = gr.Slider(0.3, 1.0, value=0.7, step=0.05, label="Expressividade")
        pausa_frase = gr.Slider(0.0, 1.0, value=0.25, step=0.05, label="Pausa entre frases (s)")
        pausa_paragrafo = gr.Slider(0.2, 3.0, value=0.9, step=0.1, label="Pausa entre parágrafos (s)")
    botao = gr.Button("Narrar", variant="primary")
    status = gr.Textbox(label="Andamento", interactive=False)
    saida = gr.File(label="Narração (WAV)")

    botao.click(
        narrar,
        inputs=[vozes, texto, velocidade, expressividade, pausa_frase, pausa_paragrafo],
        outputs=[status, saida],
        api_name="narrar",
    )

_, _, link = demo.queue(default_concurrency_limit=1, max_size=20).launch(
    share=True, prevent_thread_lock=True, show_error=True, quiet=True)

abrir = f"{PAGINA}?servidor={quote(link or '', safe='')}"
try:
    from IPython.display import HTML, display
    display(HTML(
        f'<div style="font:16px sans-serif;padding:16px;border:2px solid #b4442b;border-radius:12px;max-width:640px">'
        f'<b style="font-size:20px">✅ Servidor pronto!</b><br><br>'
        f'<a href="{abrir}" target="_blank" style="font-size:20px;color:#b4442b">👉 Clique aqui para abrir o seu narrador já conectado</a><br><br>'
        f'Deixe esta aba do Colab aberta enquanto usa. Link do servidor: <code>{link}</code></div>'))
except Exception:  # noqa: BLE001
    pass
print("\nServidor pronto! Abra:", abrir, "\n")
demo.block_thread()
